In [1]:
import pandas as pd
import numpy as np

In [10]:
df = pd.read_parquet('BTC_1m_OHLCV.parquet')
df.head()

,datetime,timestamp_ms,open,high,low,close,volume,turnover
0,2020-03-25 10:36:00+00:00,1585132560000,6500.0,6500.0,6500.0,6500.0,0.001,6.5
1,2020-03-25 10:37:00+00:00,1585132620000,6500.0,6500.0,6500.0,6500.0,0.000,0.0
2,2020-03-25 10:38:00+00:00,1585132680000,6500.0,6500.0,6500.0,6500.0,0.000,0.0
3,2020-03-25 10:39:00+00:00,1585132740000,6500.0,6500.0,6500.0,6500.0,0.000,0.0
4,2020-03-25 10:40:00+00:00,1585132800000,6500.0,6500.0,6500.0,6500.0,0.000,0.0


In [13]:
df = pd.read_parquet("BTC_1m_OHLCV.parquet")

df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
df = df.sort_values("datetime").set_index("datetime")

df = df[["open", "high", "low", "close", "volume", "turnover"]].astype(float)

df["r"] = np.log(df["close"]).diff()
df["r2"] = df["r"] ** 2
df["rq"] = df["r"] ** 4

h = pd.DataFrame(index=df.resample("1h").size().index)

h["rv"] = df["r2"].resample("1h").sum()
h["log_rv"] = np.log(h["rv"] + 1e-12)

h["rq"] = df["rq"].resample("1h").sum()
h["ret"] = np.log(df["close"].resample("1h").last()).diff()
h["abs_ret"] = h["ret"].abs()

ohlcv = df.resample("1h").agg({
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume": "sum",
    "turnover": "sum"
})

h["hl_range"] = np.log(ohlcv["high"] / ohlcv["low"])
h["oc_range"] = np.abs(np.log(ohlcv["close"] / ohlcv["open"]))

h["log_volume"] = np.log1p(ohlcv["volume"])
h["log_turnover"] = np.log1p(ohlcv["turnover"])
h["log_dollar_volume"] = np.log1p(ohlcv["close"] * ohlcv["volume"])

x = pd.DataFrame(index=h.index)

base_features = [
    "log_rv",
    "rq",
    "ret",
    "abs_ret",
    "hl_range",
    "oc_range",
    "log_volume",
    "log_turnover",
    "log_dollar_volume"
]

for col in base_features:
    x[f"{col}_lag1"] = h[col].shift(1)

for lag in [2, 3, 6, 12, 24]:
    x[f"log_rv_lag{lag}"] = h["log_rv"].shift(lag)
    x[f"ret_lag{lag}"] = h["ret"].shift(lag)
    x[f"log_volume_lag{lag}"] = h["log_volume"].shift(lag)

for window in [3, 6, 12, 24, 72, 168]:
    x[f"log_rv_mean_{window}h"] = h["log_rv"].shift(1).rolling(window).mean()
    x[f"log_rv_std_{window}h"] = h["log_rv"].shift(1).rolling(window).std()
    x[f"abs_ret_mean_{window}h"] = h["abs_ret"].shift(1).rolling(window).mean()
    x[f"log_volume_mean_{window}h"] = h["log_volume"].shift(1).rolling(window).mean()

dataset = x.copy()

dataset["target_log_rv"] = h["log_rv"]

dataset = dataset.replace([np.inf, -np.inf], np.nan).dropna()

X = dataset.drop(columns="target_log_rv")
y = dataset["target_log_rv"]

dataset.to_parquet("BTC_1h_logRV_dataset.parquet")

In [14]:
dataset.shape

(53428, 49)